# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요 (저장소 클론 후 경로·모듈 로드)
# 주의: Colab에서는 GitHub 저장소 URL과 Personal Access Token을 반드시 입력해야 합니다.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from getpass import getpass

    git_url = input("GitHub 저장소 URL (예: github.com/USERNAME/mnist-lab.git): ").strip()
    token = getpass("GitHub Personal Access Token (private 저장소인 경우): ")

    # URL 마지막 경로를 저장소 폴더명으로 사용합니다. (예: .../mnist-lab.git -> mnist-lab)
    repo_name = Path(git_url.rstrip("/")).name
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]

    !git clone https://{token}@{git_url}
    os.chdir(repo_name)
    sys.path.insert(0, str(Path.cwd() / "src"))
else:
    sys.path.insert(0, "./src")

## 2. 데이터 로드

In [ ]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [ ]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_relu.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


## 4. 모델·옵티마이저 생성 및 학습

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

EPOCHS = 20
BATCH_SIZE = 128
DROPOUT_RATIO = 0.5

experiments = [
    {"name": "2 hidden layers", "hidden_sizes": (512, 256)},
    {"name": "3 hidden layers", "hidden_sizes": (512, 256, 128)},
    {"name": "4 hidden layers", "hidden_sizes": (512, 256, 128, 64)},
]

results = []

for config in experiments:
    print(f"Training {config['name']} {config['hidden_sizes']} | epochs={EPOCHS}, batch_size={BATCH_SIZE}, dropout={DROPOUT_RATIO}")
    model = NeuralNetwork(
        use_batchnorm=True,
        use_dropout=True,
        dropout_ratio=DROPOUT_RATIO,
        hidden_sizes=config["hidden_sizes"],
    )
    optimizer = Adam(lr=0.001)
    loss_history = train(model, optimizer, x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)
    results.append({
        "name": config["name"],
        "hidden_sizes": config["hidden_sizes"],
        "model": model,
        "loss_history": loss_history,
    })
    print(f"  final loss: {loss_history[-1]:.4f}\n")


## 5. 평가 및 손실 커브

In [ ]:
import matplotlib.pyplot as plt
from training import evaluate

if "results" not in globals() or len(results) == 0:
    raise RuntimeError("먼저 4번 학습 셀을 실행해서 results를 만들어 주세요.")

for result in results:
    acc, n_params = evaluate(result["model"], x_test, y_test)
    result["accuracy"] = acc
    result["n_params"] = n_params

print("Model comparison: 2 vs 3 vs 4 hidden layers")
print("-" * 72)
print(f"{'Model':<18} {'Hidden sizes':<26} {'Accuracy':>10} {'Params':>12}")
print("-" * 72)
for result in results:
    hidden_text = str(result["hidden_sizes"])
    print(f"{result['name']:<18} {hidden_text:<26} {result['accuracy']:>9.2f}% {result['n_params']:>12,}")

best_result = max(results, key=lambda item: item["accuracy"])
print("-" * 72)
print(f"Best accuracy: {best_result['name']} {best_result['hidden_sizes']} -> {best_result['accuracy']:.2f}%")

plt.figure(figsize=(8, 5))
for result in results:
    epochs = range(1, len(result["loss_history"]) + 1)
    plt.plot(epochs, result["loss_history"], marker="o", label=f"{result['name']} {result['hidden_sizes']}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss by Hidden Layer Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

names = [result["name"] for result in results]
accuracies = [result["accuracy"] for result in results]

plt.figure(figsize=(7, 5))
bars = plt.bar(names, accuracies, color=["#4C78A8", "#F58518", "#54A24B"])
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Comparison")
upper = max(100, max(accuracies) + 3)
plt.ylim(0, upper)

for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + upper * 0.01,
        f"{acc:.2f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()
